# ML Engineer notebook: Customers Like You

**Tanay Mihani (ML Engineer).** The data side (cleaning, split, matrix) is Achintya's, in `Data_Engineer.ipynb`.

This notebook takes the training matrix Achintya built and does ML1 to ML7. Run `Data_Engineer.ipynb` first so `data/processed/` exists, then run this one top to bottom from the repo root.

1. Setup
2. ML1: method decision (written before fitting anything)
3. Load the matrix and check it matches the matrix report
4. Size of the compressed version at each k
5. ML2 onwards: next

## 1. Setup

In [1]:
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp

SEED = 42                      # used for every random step (R6)
rng = np.random.default_rng(SEED)
PROC = "data/processed"        # written by Data_Engineer.ipynb
K_GRID = [10, 20, 50, 100, 200]

## 2. ML1: Method decision

I'm writing this before fitting anything, so the choice isn't influenced by the results.

**Main method: Truncated SVD.** PCA gets run later on the same matrix with the same k, as the comparison for ML5.

Why SVD:

1. **The matrix is almost all zeros.** It's 5,456 customers × 3,680 products, 1/0 "ever bought", and 97.95% of the cells are 0. Stored sparse it's about 5 MB, dense it would be 160.6 MB. `TruncatedSVD` works on the sparse matrix directly and never has to fill in the zeros.
2. **PCA centres first, and that breaks the sparsity.** Subtracting each product's average turns every 0 into a small negative number. scikit-learn 1.4+ can do the centring implicitly with the `arpack` solver so it won't crash, but it's still working on what is effectively a dense matrix.
3. **SVD's components fit the job.** For "customers who bought this also bought", I want products that get bought by the same customers. SVD on the raw matrix captures that co-buying directly. PCA's components describe how customers differ from the average customer, which is a less direct fit for recommending products.
4. **One fit gives both sides.** SVD gives customer factors (U·S) and product factors (V) at the same time: product factors for the similar-products box, customer factors for the customer map.

Where PCA might do better: it reports explained variance, which is an easier number to give Sarah than reconstruction error, and for the customer map, centring puts the average customer in the middle of the plot. I'll check this in ML5 with my own numbers instead of assuming it.

**Prediction to check later:** SVD's first component will mostly measure how much a customer buys overall (big buyers get big values), not a particular kind of product.

### How I'll choose k (also decided before fitting)

Candidates: k = 10, 20, 50, 100, 200. I'll use two kinds of evidence:

1. **Reconstruction error vs k**, ‖A − A_k‖ / ‖A‖ (Frobenius norm). Look for the elbow, where adding more components stops reducing the error much.
2. **Hit rate at 10 vs k.** Pick the smallest k within 1 percentage point of the best hit rate, because a smaller k is cheaper to store and faster to run.

The test months can't be used to choose k, because then the final test score would be optimistic (R2). So k gets chosen on a **validation split inside the training period**: learn from everything before 9 Aug 2011, check against new products bought from 9 Aug to 9 Oct 2011. That's the same two-month setup as the real test. Then I refit on the full training matrix with the chosen k and score the test set once. This needs the matrix rebuilt with an earlier cutoff, which I've asked Achintya to support in `build_matrix.py`.

### How hit rate gets counted

- **Test customers:** all 1,919 customers in `heldout.csv`. These are customers in the training matrix who bought at least one product they had never bought before in the test months.
- **Hit:** at least one of the customer's held-out products is in their top 10. Products they already bought in training are never recommended.
- **Products outside the matrix:** about 9.5% of held-out purchases are products that aren't in the matrix (new products, or ones bought by fewer than 10 customers). Neither the model nor the baseline can ever recommend them. I keep them in, and they count as misses for both, so the comparison stays fair. I report how many customers this affects (cell below).
- **Baseline (R3):** the 10 products with the most distinct buyers in training (`n_customers` in `products.csv`), skipping any the customer already bought. I use distinct buyers rather than units so one wholesaler's huge order can't make something a "best seller".

## 3. Load the matrix and check it

Customer IDs never get loaded here (R4). From `customers_internal.csv` I only read the row index and the country.

In [2]:
A = sp.load_npz(f"{PROC}/A_train.npz").tocsr()
products = pd.read_csv(f"{PROC}/products.csv", dtype={"StockCode": str})
customers = pd.read_csv(f"{PROC}/customers_internal.csv", usecols=["row", "Country"])
heldout = pd.read_csv(f"{PROC}/heldout.csv", dtype={"StockCode": str})
heldout["prod_col"] = heldout["prod_col"].astype("Int64")
with open(f"{PROC}/matrix_report.json") as f:
    report = json.load(f)

n_cust, n_prod = A.shape
dense_mb = n_cust * n_prod * 8 / 1e6
sparse_mb = (A.data.nbytes + A.indices.nbytes + A.indptr.nbytes) / 1e6
print(f"Matrix: {n_cust:,} customers x {n_prod:,} products, {A.nnz:,} non-zeros, "
      f"{1 - A.nnz / (n_cust * n_prod):.2%} zeros")
print(f"Values stored: {np.unique(A.data)}  (should only be 1)")
print(f"Memory: dense {dense_mb:.1f} MB vs sparse {sparse_mb:.2f} MB")

# Same numbers as Achintya's matrix report, and the index files line up with the matrix
assert (n_cust, n_prod, A.nnz) == (report["customers"], report["products"], report["non_zeros"])
assert len(products) == n_prod and len(customers) == n_cust
print("Matches matrix_report.json")

Matrix: 5,456 customers x 3,680 products, 411,136 non-zeros, 97.95% zeros
Values stored: [1.]  (should only be 1)
Memory: dense 160.6 MB vs sparse 4.96 MB
Matches matrix_report.json


In [3]:
test_rows = heldout["cust_row"].unique()
in_mat = heldout[heldout["in_matrix"]]
reachable = in_mat["cust_row"].unique()
per_cust = heldout.groupby("cust_row").size()

print(f"Test customers: {len(test_rows):,}")
print(f"Held-out purchases: {len(heldout):,}, of which {(~heldout['in_matrix']).mean():.1%} are products outside the matrix")
print(f"Customers with at least one held-out product that can be recommended: "
      f"{len(reachable):,} ({len(reachable) / len(test_rows):.1%})")
print(f"Held-out products per test customer: median {per_cust.median():.0f}, max {per_cust.max():,}")

# Leak check: no held-out purchase should already be a 1 in the training matrix
already = A[in_mat["cust_row"].to_numpy(), in_mat["prod_col"].to_numpy(dtype=int)]
assert np.asarray(already).sum() == 0
print("Leak check passed: no held-out purchase appears in the training matrix")

Test customers: 1,919
Held-out purchases: 49,214, of which 9.5% are products outside the matrix
Customers with at least one held-out product that can be recommended: 1,903 (99.2%)
Held-out products per test customer: median 17, max 462
Leak check passed: no held-out purchase appears in the training matrix


## 4. How big would the compressed version be?

After compressing to k components I keep two small tables: customer factors (customers × k) and product factors (products × k), each number taking 8 bytes. This is just arithmetic, no fitting.

In [4]:
print(f"Dense matrix {dense_mb:.1f} MB, sparse matrix {sparse_mb:.2f} MB\n")
for k in K_GRID:
    mb = (n_cust + n_prod) * k * 8 / 1e6
    print(f"k = {k:>3}: factors {mb:5.2f} MB  |  {dense_mb / mb:4.0f}x smaller than dense  |  "
          f"{mb / sparse_mb:.1f}x the sparse matrix")

Dense matrix 160.6 MB, sparse matrix 4.96 MB

k =  10: factors  0.73 MB  |   220x smaller than dense  |  0.1x the sparse matrix
k =  20: factors  1.46 MB  |   110x smaller than dense  |  0.3x the sparse matrix
k =  50: factors  3.65 MB  |    44x smaller than dense  |  0.7x the sparse matrix
k = 100: factors  7.31 MB  |    22x smaller than dense  |  1.5x the sparse matrix
k = 200: factors 14.62 MB  |    11x smaller than dense  |  2.9x the sparse matrix


Every k on the list is at least 10x smaller than the dense matrix, so the size target in the README holds whatever k I pick. The sparse file is already small, though, and from k = 100 up the factors are bigger than it. So the real saving is against the full customer × product score table the old prototype built (dense size), not against the sparse file. That goes in the memo.

## 5. Next: ML2

Fit `TruncatedSVD` for each k in `K_GRID` and plot reconstruction error against k. The validation hit rate for choosing k comes once the validation matrix exists.